# Simple React Agent

A minimal LangChain agent that reasons step by step and picks between a web search tool and a weather lookup tool to answer a question.

## Setup

In [7]:
import os
import requests
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_classic.agents import create_react_agent, AgentExecutor

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

## Tools

In [9]:
search_tool = TavilySearch(max_results=2)


@tool
def get_weather_data(city: str) -> str:
    """Fetch current weather information for a city."""
    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )
    response = requests.get(url)
    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    print(data)

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        
        f"Humidity: {data['current']['humidity']}%"
    )


tools = [search_tool, get_weather_data]

## LLM

In [10]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
)

## Prompt

In [11]:
prompt = PromptTemplate.from_template(
    "Answer the following questions as best you can. You have access to the following tools:\n\n"
    "{tools}\n\n"
    "Use the following format:\n\n"
    "Question: the input question you must answer\n"
    "Thought: you should always think about what to do\n"
    "Action: the action to take, should be one of [{tool_names}]\n"
    "Action Input: the input to the action\n"
    "Observation: the result of the action\n"
    "... (this Thought/Action/Action Input/Observation can repeat N times)\n"
    "Thought: I now know the final answer\n"
    "Final Answer: the final answer to the original input question\n\n"
    "Begin!\n\n"
    "Question: {input}\n"
    "Thought:{agent_scratchpad}"
)

## Agent

In [12]:
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
)

## Run

In [13]:
response = agent_executor.invoke({
    "input": "Find the current weather in Pokhara."
})

print(response["output"])



> Entering new AgentExecutor chain...
Thought: The user is asking for the current weather in Pokhara. I have a tool `get_weather_data` that can fetch current weather information for a city. I should use this tool.
Action: get_weather_data
Action Input: PokharaCould not fetch weather data for PokharaAction: tavily_search
Action Input: current weather in Pokhara{'query': 'current weather in Pokhara', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'title': 'Weather in Pokhara', 'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'Pokhara', 'region': '', 'country': 'Nepal', 'lat': 28.2333, 'lon': 83.9833, 'tz_id': 'Asia/Kathmandu', 'localtime_epoch': 1787707942, 'localtime': '2026-08-26 07:17'}, 'current': {'last_updated_epoch': 1787706900, 'last_updated': '2026-08-26 07:00', 'temp_c': 20.9, 'temp_f': 69.7, 'is_day': 1, 'condition': {'text': 'Light rain shower', 'icon': '//cdn.weatherapi.com/weather/64x64/day/353.png', 'code': 1240}, 'wind_mph